In [1]:
from sklearn.feature_extraction.text import CountVectorizer
v = CountVectorizer()
v.fit(["Thor Hathodawala is looking for job"])
v.vocabulary_

{'thor': 5, 'hathodawala': 1, 'is': 2, 'looking': 4, 'for': 0, 'job': 3}

In [5]:
v = CountVectorizer(ngram_range=(1,2))
v.fit(["Thor Hathodawala is looking for job"])
v.vocabulary_

{'thor': 9,
 'hathodawala': 2,
 'is': 4,
 'looking': 7,
 'for': 0,
 'job': 6,
 'thor hathodawala': 10,
 'hathodawala is': 3,
 'is looking': 5,
 'looking for': 8,
 'for job': 1}

In [6]:
corpus = [
    "Thor ate pizza",
    "Loki is tall",
    "Loki is eating pizza"
]

In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text)
    filterd_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filterd_tokens.append(token.lemma_)
    
    return " ".join(filterd_tokens)

In [14]:
corpus_processed = [
    preprocess(text) for text in corpus
]
corpus_processed

['thor eat pizza', 'Loki tall', 'Loki eat pizza']

In [21]:
import pandas as pd

df = pd.read_json('news_dataset.json')
print(df.shape)
df.head()

(12695, 2)


,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [22]:
df.category.value_counts()

category
BUSINESS    4254
SPORTS      4167
CRIME       2893
SCIENCE     1381
Name: count, dtype: int64

In [23]:
min_sample = 1381

df_business = df[df.category == 'BUSINESS'].sample(min_sample,random_state=42)
df_sports = df[df.category == 'SPORTS'].sample(min_sample,random_state=42)
df_crime = df[df.category == 'CRIME'].sample(min_sample,random_state=42)
df_science = df[df.category == 'SCIENCE'].sample(min_sample,random_state=42)

In [24]:
df_balanced = pd.concat([df_business,df_sports,df_crime,df_science], axis = 0)
df_balanced.category.value_counts()

category
BUSINESS    1381
SPORTS      1381
CRIME       1381
SCIENCE     1381
Name: count, dtype: int64

In [25]:
df_balanced['category_num'] = df_balanced['category'].map({
    'BUSINESS' : 0,
    'SPORTS' : 1,
    'CRIME' : 2,
    'SCIENCE' : 3,
})

In [26]:
df_balanced.head()

,text,category,category_num
594,How to Develop the Next Generation of Innovato...,BUSINESS,0
3093,"Madoff Victims' Payout Nears $7.2 Billion, Tru...",BUSINESS,0
7447,Bay Area Floats 'Sanctuary In Transit Policy' ...,BUSINESS,0
10388,Microsoft Agrees To Acquire LinkedIn For $26.2...,BUSINESS,0
1782,"Inside A Legal, Multibillion Dollar Weed Market",BUSINESS,0


In [28]:
df_balanced['preprocessed_text'] = df_balanced['text'].apply(preprocess)


In [29]:
df_balanced.head()

,text,category,category_num,preprocessed_text
594,How to Develop the Next Generation of Innovato...,BUSINESS,0,develop Generation Innovators stop treat way g...
3093,"Madoff Victims' Payout Nears $7.2 Billion, Tru...",BUSINESS,0,Madoff Victims Payout near $ 7.2 billion Trust...
7447,Bay Area Floats 'Sanctuary In Transit Policy' ...,BUSINESS,0,Bay Area Floats Sanctuary Transit Policy prote...
10388,Microsoft Agrees To Acquire LinkedIn For $26.2...,BUSINESS,0,Microsoft agree acquire linkedin $ 26.2 billio...
1782,"Inside A Legal, Multibillion Dollar Weed Market",BUSINESS,0,inside Legal Multibillion Dollar Weed Market


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [32]:
X_train, X_test, y_train, y_test = train_test_split(df_balanced.preprocessed_text, df_balanced.category_num, test_size=0.2 , random_state=42, stratify=df_balanced.category_num)

In [33]:
print(X_train.shape)
X_train.head()

(4419,)


6414     Arby employee keep Job refuse serve Police Arb...
1318     colorful NASA image show Pluto Psychedelic vib...
4170     woman Business Q&A Sophie Delafontaine Artisti...
11310    5 Formalized Referral Systems grow sale potent...
4188     Hawaii Kilauea Volcano see Mesmerizing River L...
Name: preprocessed_text, dtype: object

In [34]:
y_train.value_counts()

category_num
0    1105
3    1105
1    1105
2    1104
Name: count, dtype: int64

In [39]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,1))),
    ('model', MultinomialNB())
])
clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)
print(clf.score(X_test,y_test))
print(classification_report(y_test,y_pred))

0.8886877828054298
              precision    recall  f1-score   support

           0       0.85      0.90      0.87       276
           1       0.91      0.88      0.89       276
           2       0.88      0.94      0.91       277
           3       0.92      0.84      0.88       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105



In [40]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('model', MultinomialNB())
])
clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)
print(clf.score(X_test,y_test))
print(classification_report(y_test,y_pred))

0.8868778280542986
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       276
           1       0.92      0.87      0.89       276
           2       0.88      0.94      0.91       277
           3       0.93      0.82      0.87       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105

